# SEMIR LiTS Reproduction — v4 oracle-first corrected

This version fixes the failure mode seen in `semir_lits_v3_run_summary.json`: the few-shot search selected an over-compressed graph with ~129 supernodes, oracle Dice ~0.03, and ~33% tumor deletion.

The core correction is: **never let a candidate win unless the graph can still recover the tumor**. Compression is a secondary objective after oracle Dice and tumor-retention are acceptable.


## 1. Setup

In [1]:

import numpy as np
import os, re, time, json, math, random
import fastloops

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"

# Default HU window. The search below can be run under either [-50,250] or [0,200].
# Keep this global value synchronized with the selected search result before building graphs.
HU_MIN, HU_MAX = -50, 250

# These are overwritten by the oracle-first few-shot search in Section 4.
MERGE_DIST = 8       # ψ
CUT_DIST = 51        # α
DELETE_SMALL = 0     # β_min -- start conservative; tumor rind nodes must survive
DELETE_LARGE_FRAC = 1.00  # β_max = int(n_vox ** DELETE_LARGE_FRAC); 1.00 disables large-node deletion
VALUE_MIN = 0        # m_min -- disable intensity deletion until oracle is high
VALUE_MAX = 255      # m_max

# A strict 0.50 overlap threshold often yields almost no positive tumor supernodes
# when the graph is still imperfect. Sweep this after oracle is acceptable.
OVERLAP_THRESHOLD = 0.05

np.random.seed(42)
random.seed(42)

print("fastloops loaded")
print("Initial params will be overwritten by few-shot search:")
print(f"  HU=[{HU_MIN},{HU_MAX}]  ψ={MERGE_DIST}  α={CUT_DIST}")
print(f"  β_min={DELETE_SMALL}  β_max=n_vox**{DELETE_LARGE_FRAC}  m=[{VALUE_MIN},{VALUE_MAX}]")
print(f"  overlap_threshold={OVERLAP_THRESHOLD}")


fastloops loaded
Initial params will be overwritten by few-shot search:
  HU=[-50,250]  ψ=8  α=51
  β_min=0  β_max=n_vox**1.0  m=[0,255]
  overlap_threshold=0.05


## 2. Discover LiTS Volumes

In [2]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

N_WORKERS = min(mp.cpu_count(), 32)  # use up to 32 cores
print(f"Available CPUs: {mp.cpu_count()}, using {N_WORKERS} workers")

def _check_tumor(vid):
    seg = np.load(os.path.join(DATA_ROOT, 'seg', f'segmentation-{vid}.npy'))
    return vid if (seg == 2).sum() > 0 else None

def discover_volumes():
    ct_dir = os.path.join(DATA_ROOT, 'ct')
    vids = []
    for f in sorted(os.listdir(ct_dir)):
        m = re.match(r'volume-(\d+)\.npy', f)
        if m:
            vid = int(m.group(1))
            if os.path.exists(os.path.join(DATA_ROOT, 'seg', f'segmentation-{vid}.npy')):
                vids.append(vid)
    # Parallel tumor check
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        results = list(pool.map(_check_tumor, vids))
    return sorted([v for v in results if v is not None])

t0 = time.time()
all_vids = discover_volumes()
print(f"Found {len(all_vids)} LiTS volumes with tumor ({time.time()-t0:.1f}s)")

np.random.seed(42)
perm = np.random.permutation(len(all_vids))
n_train = int(0.7 * len(all_vids))
n_val = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_train]])
val_ids = sorted([all_vids[i] for i in perm[n_train:n_train + n_val]])
test_ids = sorted([all_vids[i] for i in perm[n_train + n_val:]])
ordered = train_ids + val_ids + test_ids
print(f"Split: {len(train_ids)} train / {len(val_ids)} val / {len(test_ids)} test")


Available CPUs: 128, using 32 workers


Found 118 LiTS volumes with tumor (0.3s)
Split: 82 train / 17 val / 19 test


## 3. Rust contraction/deletion helpers

The old notebook built all graphs once before the few-shot search. That is expensive and misleading. In this version the all-volume graph construction happens only **after** `Θopt` is selected.


In [3]:
def load_and_convert(vid, hu_min=None, hu_max=None):
    """Load raw CT and convert to uint8 with the requested HU window."""
    hu_min = HU_MIN if hu_min is None else hu_min
    hu_max = HU_MAX if hu_max is None else hu_max
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, hu_min, hu_max)
    ct_u8 = ((ct_u8 - hu_min) / (hu_max - hu_min) * 255).round().astype(np.uint8)
    ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])  # (D, H, W, 1)
    return ct, seg, ct_u8


def oracle_dice(labels_np, seg):
    """Quality ceiling of the graph minor.

    A perfect supernode classifier marks every surviving supernode that contains
    any tumor voxel as foreground. If this score is low, the coarsener has already
    destroyed the segmentation problem.
    """
    flat = labels_np.ravel()
    gt = (seg.ravel() == 2).astype(np.float64)
    gt_total = int(gt.sum())
    valid = flat >= 0
    if gt_total == 0 or not valid.any():
        return 0.0, 0, 100.0 if gt_total > 0 else 0.0
    max_id = int(flat[valid].max())
    tc = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
    tumor_sids = np.where(tc > 0)[0]
    lut = np.zeros(max_id + 1, dtype=np.int32)
    lut[tumor_sids] = 1
    pred = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
    gt_mask = seg == 2
    inter = int((pred & gt_mask).sum())
    dice = 2.0 * inter / (pred.sum() + gt_mask.sum() + 1e-8)
    deleted_tumor = int(gt[~valid].sum())
    deleted_pct = deleted_tumor / max(gt_total, 1) * 100.0
    return dice, len(tumor_sids), deleted_pct


def compute_intensity_std(labels_np, ct_u8):
    """Per-supernode intensity std in [0,1], indexed by supernode ID."""
    flat = labels_np.ravel()
    valid = flat >= 0
    if not valid.any():
        return np.array([], dtype=np.float32)
    max_id = int(flat[valid].max())
    vals = ct_u8[..., 0].ravel().astype(np.float64) / 255.0
    counts = np.bincount(flat[valid], minlength=max_id + 1).astype(np.float64)
    sums = np.bincount(flat[valid], weights=vals[valid], minlength=max_id + 1)
    sq_sums = np.bincount(flat[valid], weights=vals[valid] ** 2, minlength=max_id + 1)
    mean = sums / np.maximum(counts, 1.0)
    var = sq_sums / np.maximum(counts, 1.0) - mean ** 2
    return np.sqrt(np.maximum(var, 0.0)).astype(np.float32)


def boundary_mask_6conn(mask):
    """6-connected binary boundary without scipy dependency."""
    mask = mask.astype(bool)
    b = np.zeros_like(mask, dtype=bool)
    for axis in range(3):
        lo = [slice(None)] * 3
        hi = [slice(None)] * 3
        lo[axis] = slice(0, -1)
        hi[axis] = slice(1, None)
        diff = mask[tuple(lo)] != mask[tuple(hi)]
        b[tuple(lo)] |= diff
        b[tuple(hi)] |= diff
    return b & mask


def supernode_boundary_mask(labels_np):
    """Voxels adjacent to a different surviving supernode."""
    valid = labels_np >= 0
    b = np.zeros_like(labels_np, dtype=bool)
    for axis in range(3):
        lo = [slice(None)] * 3
        hi = [slice(None)] * 3
        lo[axis] = slice(0, -1)
        hi[axis] = slice(1, None)
        a = labels_np[tuple(lo)]
        c = labels_np[tuple(hi)]
        v = valid[tuple(lo)] & valid[tuple(hi)]
        diff = (a != c) & v
        b[tuple(lo)] |= diff
        b[tuple(hi)] |= diff
    return b & valid


def boundary_dice(labels_np, seg, target_label=2):
    """SEMIR few-shot objective: DSC(supernode boundaries, target GT boundary)."""
    gt_b = boundary_mask_6conn(seg == target_label)
    sn_b = supernode_boundary_mask(labels_np)
    denom = gt_b.sum() + sn_b.sum()
    if denom == 0:
        return 0.0
    return 2.0 * int((gt_b & sn_b).sum()) / (denom + 1e-8)


def run_minor(ct_u8, n_vox, params):
    """One wrapper around the Rust coarsener."""
    beta_max = int(n_vox ** float(params["beta_max_frac"]))
    return fastloops.merge_and_cut(
        ct_u8,
        merge_distance=int(params["psi"]),
        cut_distance=int(params["alpha"]),
        delete_small_node_max_size=int(params["beta_min"]),
        delete_large_node_min_size=beta_max,
        delete_value_min=int(params["m_min"]),
        delete_value_max=int(params["m_max"]),
        connectivity="faces",
    )

print("Helper functions defined.")


Helper functions defined.


In [4]:

# Optional baseline diagnostic only. Keep disabled for normal reproduction runs.
# The actual full graph build happens after the oracle-first few-shot search.
RUN_BASELINE_DIAGNOSTIC = False

if RUN_BASELINE_DIAGNOSTIC:
    def _build_minor_for_vid(vid):
        """Worker: build graph minor for one volume using the initial fallback params."""
        import fastloops
        ct_raw, seg, ct_u8 = load_and_convert(vid)
        n_vox = ct_raw.size
        t0 = time.time()
        raw_nf, raw_ei, raw_ef, raw_labels, raw_adj = fastloops.merge_and_cut(
            ct_u8, merge_distance=MERGE_DIST, cut_distance=CUT_DIST,
            delete_small_node_max_size=DELETE_SMALL,
            delete_large_node_min_size=int(n_vox ** DELETE_LARGE_FRAC),
            delete_value_min=VALUE_MIN, delete_value_max=VALUE_MAX,
            connectivity='faces',
        )
        dt = time.time() - t0
        labels_np = np.asarray(raw_labels)
        od, t_sn, del_pct = oracle_dice(labels_np, seg)
        return vid, raw_nf.shape[0], raw_ei.shape[1], od, del_pct, dt

    t0 = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        baseline_results = list(pool.map(_build_minor_for_vid, ordered))
    for vid, n_sn, n_edges, od, del_pct, dt in baseline_results:
        print(f"vol-{vid}: {n_sn:,} SN  {n_edges:,} edges  oracle={od:.4f}  del={del_pct:.1f}%  {dt:.1f}s")
    print(f"Baseline diagnostic done in {time.time()-t0:.1f}s")
else:
    print("Skipping pre-search all-volume graph build. This is intentional.")


Skipping pre-search all-volume graph build. This is intentional.


## 3b. Optional HU-window diagnostic

Run this only if the oracle-first search cannot find a candidate with acceptable oracle Dice. The goal is to compare whether `[0,200]` or `[-50,250]` gives cleaner tumor-preserving supernodes under conservative no-deletion settings.


In [5]:
RUN_HU_DIAGNOSTIC = False

if RUN_HU_DIAGNOSTIC:
    hu_configs = [
        (0, 200, "HU [0, 200] liver window"),
        (-50, 250, "HU [-50, 250] wider window"),
    ]
    test_vids = ordered[:min(20, len(ordered))]
    for hu_min, hu_max, label in hu_configs:
        oracles, sns, dels = [], [], []
        for vid in test_vids:
            ct_raw = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
            seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
            n_vox = ct_raw.size
            ct_u8 = np.clip(ct_raw, hu_min, hu_max)
            ct_u8 = ((ct_u8 - hu_min) / (hu_max - hu_min) * 255).round().astype(np.uint8)
            ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])
            nf, ei, ef, labels, adj = fastloops.merge_and_cut(
                ct_u8,
                merge_distance=8,
                cut_distance=64,
                delete_small_node_max_size=0,
                delete_large_node_min_size=n_vox,
                delete_value_min=0,
                delete_value_max=255,
                connectivity="faces",
            )
            od, t_sn, del_pct = oracle_dice(np.asarray(labels), seg)
            oracles.append(od); sns.append(nf.shape[0]); dels.append(del_pct)
        print(f"{label}:")
        print(f"  Oracle Dice: {np.mean(oracles):.4f} +/- {np.std(oracles):.4f}")
        print(f"  Supernodes:  {np.mean(sns):,.0f}")
        print(f"  Tumor deleted: {np.mean(dels):.2f}%")
        print()
else:
    print("Skipping optional HU diagnostic.")


Skipping optional HU diagnostic.


## 4. Oracle-first few-shot SEMIR parameter search

The previous run selected a candidate with too few nodes and near-zero oracle Dice. This corrected search uses hard/soft oracle-first scoring:

1. Reject or heavily demote candidates with low oracle Dice.
2. Reject candidates that delete tumor voxels.
3. Only then prefer paper-like compression.
4. Use a tumor-boundary band for boundary Dice so global background boundaries do not dominate.


In [6]:

# -------------------------
# Oracle-first few-shot parameter search — PARALLELIZED
# -------------------------
N_FEW = min(5, len(train_ids))
N_RANDOM = 250            # increase to 1000+ after the logic is validated
TOP_K_PRINT = 20
few_vids = train_ids[:N_FEW]
print(f"Few-shot volumes: {few_vids}")

# Search-space philosophy:
# - Keep β_min small so tumor rind nodes survive.
# - Disable intensity deletion first: m=[0,255]. Add intensity windows only after oracle is high.
# - Keep ψ moderate. ψ=42 over-compressed the graph in the previous run.
# - Set α as a multiple of ψ. α must be above ψ, but α=253 on uint8 effectively cuts almost nothing.
psi_values = [3, 4, 5, 6, 8, 10, 12, 15, 18, 22, 26]
beta_min_values = [0, 1, 2, 3, 5, 8, 13]
beta_max_frac_values = [0.96, 1.00]  # 1.00 effectively disables large-node deletion
alpha_multipliers = [2.5, 3.5, 5.0, 7.0, 10.0]
m_ranges = [(0, 255)]

# Deterministic grid candidates.
candidates = []
for psi in psi_values:
    for mult in alpha_multipliers:
        alpha = min(int(round(psi * mult)), 200)
        alpha = max(alpha, psi + 4)
        for beta_min in beta_min_values:
            for beta_max_frac in beta_max_frac_values:
                for m_min, m_max in m_ranges:
                    candidates.append(dict(
                        psi=int(psi), alpha=int(alpha), beta_min=int(beta_min),
                        beta_max_frac=float(beta_max_frac), m_min=int(m_min), m_max=int(m_max),
                    ))

# Add a small random tail, still constrained to tumor-preserving ranges.
rng = np.random.default_rng(42)
for _ in range(N_RANDOM):
    psi = int(rng.choice([3,4,5,6,7,8,9,10,12,14,16,18,20,22,24,26,30]))
    alpha = int(min(max(round(psi * rng.choice([2.0,2.5,3.0,3.5,4.5,6.0,8.0,10.0])), psi + 4), 220))
    beta_min = int(rng.choice([0,1,2,3,5,8,13,21]))
    beta_max_frac = float(rng.choice([0.94,0.96,0.98,1.00]))
    # Keep intensity deletion disabled in 80% of random candidates.
    if rng.random() < 0.80:
        m_min, m_max = 0, 255
    else:
        m_min = int(rng.choice([0,3,5,8,13]))
        m_max = int(rng.choice([242,250,255]))
    candidates.append(dict(psi=psi, alpha=alpha, beta_min=beta_min,
                           beta_max_frac=beta_max_frac, m_min=m_min, m_max=m_max))

# Deduplicate.
seen, uniq = set(), []
for c in candidates:
    key = tuple(c[k] for k in ['psi', 'alpha', 'beta_min', 'beta_max_frac', 'm_min', 'm_max'])
    if key not in seen:
        seen.add(key); uniq.append(c)
candidates = uniq
print(f"Evaluating {len(candidates)} candidates on {N_FEW} cases with {N_WORKERS} workers...")

# Pre-save few-shot data to /dev/shm for fast worker access.
FEW_DIR = '/dev/shm/semir_few_shot_oracle_first'
os.makedirs(FEW_DIR, exist_ok=True)
few_meta = []
for vid in few_vids:
    ct_raw, seg, ct_u8 = load_and_convert(vid)
    np.save(f'{FEW_DIR}/ct_u8_{vid}.npy', ct_u8)
    np.save(f'{FEW_DIR}/seg_{vid}.npy', seg)
    few_meta.append({'vid': vid, 'n_vox': ct_raw.size})


def _eval_candidate(args):
    """Worker: evaluate one candidate across few-shot volumes."""
    params, meta_list, few_dir = args
    import fastloops
    import numpy as _np

    def _dilate6(mask, iterations=3):
        mask = mask.astype(bool)
        out = mask.copy()
        for _ in range(iterations):
            m = out.copy()
            for ax in range(3):
                lo = [slice(None)] * 3; hi = [slice(None)] * 3
                lo[ax] = slice(0, -1); hi[ax] = slice(1, None)
                m[tuple(lo)] |= out[tuple(hi)]
                m[tuple(hi)] |= out[tuple(lo)]
            out = m
        return out

    def _boundary_dice_band(labels_np, seg):
        # Compare supernode boundaries only near the target tumor boundary.
        gt_mask = seg == 2
        valid = labels_np >= 0
        b_gt = _np.zeros_like(gt_mask, dtype=bool)
        b_sn = _np.zeros_like(labels_np, dtype=bool)
        for axis in range(3):
            lo = [slice(None)] * 3; hi = [slice(None)] * 3
            lo[axis] = slice(0, -1); hi[axis] = slice(1, None)
            d_gt = gt_mask[tuple(lo)] != gt_mask[tuple(hi)]
            b_gt[tuple(lo)] |= d_gt; b_gt[tuple(hi)] |= d_gt
            a = labels_np[tuple(lo)]; c = labels_np[tuple(hi)]
            v = valid[tuple(lo)] & valid[tuple(hi)]
            d_sn = (a != c) & v
            b_sn[tuple(lo)] |= d_sn; b_sn[tuple(hi)] |= d_sn
        band = _dilate6(b_gt, iterations=3) | gt_mask
        b_gt &= band
        b_sn &= band
        denom = b_gt.sum() + b_sn.sum()
        return 2.0 * int((b_gt & b_sn).sum()) / (denom + 1e-8) if denom > 0 else 0.0

    def _oracle_dice(labels_np, seg):
        flat = labels_np.ravel()
        gt = (seg.ravel() == 2).astype(_np.float64)
        gt_total = int(gt.sum())
        valid = flat >= 0
        if gt_total == 0 or not valid.any():
            return 0.0, 0, 100.0 if gt_total > 0 else 0.0
        max_id = int(flat[valid].max())
        tc = _np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
        tumor_sids = _np.where(tc > 0)[0]
        lut = _np.zeros(max_id + 1, dtype=_np.int32)
        lut[tumor_sids] = 1
        pred = _np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
        gt_mask = seg == 2
        inter = int((pred & gt_mask).sum())
        dice = 2.0 * inter / (pred.sum() + gt_mask.sum() + 1e-8)
        deleted_tumor = int(gt[~valid].sum())
        return dice, len(tumor_sids), deleted_tumor / max(gt_total, 1) * 100.0

    bds, ods, sns, dels, tumor_sns, edge_counts = [], [], [], [], [], []
    for m in meta_list:
        try:
            ct_u8 = _np.load(f"{few_dir}/ct_u8_{m['vid']}.npy")
            seg = _np.load(f"{few_dir}/seg_{m['vid']}.npy")
            n_vox = int(m['n_vox'])
            beta_max = int(n_vox ** float(params['beta_max_frac']))
            nf, ei, ef, labels, adj = fastloops.merge_and_cut(
                ct_u8,
                merge_distance=int(params['psi']),
                cut_distance=int(params['alpha']),
                delete_small_node_max_size=int(params['beta_min']),
                delete_large_node_min_size=beta_max,
                delete_value_min=int(params['m_min']),
                delete_value_max=int(params['m_max']),
                connectivity='faces',
            )
            labels_np = _np.asarray(labels)
            bd = _boundary_dice_band(labels_np, seg)
            od, t_sn, del_pct = _oracle_dice(labels_np, seg)
            bds.append(bd); ods.append(od); sns.append(nf.shape[0])
            dels.append(del_pct); tumor_sns.append(t_sn); edge_counts.append(ei.shape[1])
        except Exception:
            return None

    row = dict(params)
    row.update(
        boundary=float(_np.mean(bds)),
        oracle=float(_np.mean(ods)),
        sn=float(_np.mean(sns)),
        del_pct=float(_np.mean(dels)),
        tumor_sn=float(_np.mean(tumor_sns)),
        edges=float(_np.mean(edge_counts)),
    )

    # Oracle-first scoring. Candidates with low oracle cannot win.
    target_sn = 1500.0
    sn_term = -abs(_np.log((row['sn'] + 1.0) / target_sn))

    if row['oracle'] < 0.50:
        row['score'] = -1e6 + row['oracle']
    elif row['oracle'] < 0.75:
        row['score'] = -1e4 + 10.0 * row['oracle'] - 0.1 * row['del_pct']
    elif row['del_pct'] > 5.0:
        row['score'] = -1e3 + 10.0 * row['oracle'] - row['del_pct']
    else:
        row['score'] = (
            5.0 * row['oracle']
            + 1.0 * row['boundary']
            + 0.35 * sn_term
            - 0.05 * row['del_pct']
        )
    return row


t0 = time.time()
work_items = [(c, few_meta, FEW_DIR) for c in candidates]
with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    raw_rows = list(pool.map(_eval_candidate, work_items))

rows = [r for r in raw_rows if r is not None]
elapsed = time.time() - t0
print(f"Evaluated {len(rows)}/{len(candidates)} candidates in {elapsed:.1f}s "
      f"({elapsed/max(len(candidates),1):.2f}s/candidate, {N_WORKERS} workers)")

if not rows:
    raise RuntimeError('No valid SEMIR candidates were evaluated.')

# Print top candidates by oracle first, then by final score.
print("\nTop candidates by ORACLE Dice:")
print(f"{'rank':>4s} {'score':>10s} {'bd':>7s} {'oracle':>7s} {'SN':>8s} {'del%':>6s} {'tumSN':>7s}  params")
for rank, r in enumerate(sorted(rows, key=lambda r: (r['oracle'], -r['del_pct'], -abs(np.log((r['sn']+1)/1500.0)), r['boundary']), reverse=True)[:TOP_K_PRINT], start=1):
    ptxt = f"ψ={r['psi']} α={r['alpha']} βmin={r['beta_min']} βmax=n^{r['beta_max_frac']:.2f} m=[{r['m_min']},{r['m_max']}]"
    print(f"{rank:4d} {r['score']:10.2f} {r['boundary']:7.4f} {r['oracle']:7.4f} {r['sn']:8.0f} {r['del_pct']:5.1f}% {r['tumor_sn']:7.1f}  {ptxt}")

rows_by_score = sorted(rows, key=lambda r: r['score'], reverse=True)
print("\nTop candidates by FINAL oracle-first score:")
for rank, r in enumerate(rows_by_score[:TOP_K_PRINT], start=1):
    ptxt = f"ψ={r['psi']} α={r['alpha']} βmin={r['beta_min']} βmax=n^{r['beta_max_frac']:.2f} m=[{r['m_min']},{r['m_max']}]"
    print(f"{rank:4d} score={r['score']:10.2f} bd={r['boundary']:.4f} oracle={r['oracle']:.4f} SN={r['sn']:.0f} del={r['del_pct']:.1f}% tumSN={r['tumor_sn']:.1f}  {ptxt}")

# Prefer candidates satisfying minimum viability. Fall back to best oracle with a loud warning.
viable = [r for r in rows if r['oracle'] >= 0.75 and r['del_pct'] <= 5.0 and r['tumor_sn'] >= 1]
if viable:
    BEST = sorted(viable, key=lambda r: r['score'], reverse=True)[0]
    print(f"\nSelected viable Θopt from {len(viable)} viable candidates.")
else:
    BEST = sorted(rows, key=lambda r: (r['oracle'], -r['del_pct'], r['boundary']), reverse=True)[0]
    print("\nWARNING: No candidate met oracle>=0.75 and tumor deletion<=5%.")
    print("Selected the best-oracle fallback so you can inspect diagnostics, but do NOT expect GINE training to reproduce the paper.")

MERGE_DIST = int(BEST['psi'])
CUT_DIST = int(BEST['alpha'])
DELETE_SMALL = int(BEST['beta_min'])
DELETE_LARGE_FRAC = float(BEST['beta_max_frac'])
VALUE_MIN = int(BEST['m_min'])
VALUE_MAX = int(BEST['m_max'])

print(f"\nSelected Θopt:")
print(json.dumps({k: BEST[k] for k in ['psi', 'alpha', 'beta_min', 'beta_max_frac',
                                        'm_min', 'm_max', 'boundary', 'oracle', 'sn', 'del_pct', 'tumor_sn']}, indent=2))

# Clean up shared memory.
import shutil
shutil.rmtree(FEW_DIR, ignore_errors=True)


Few-shot volumes: [0, 3, 4, 5, 6]
Evaluating 997 candidates on 5 cases with 32 workers...


Evaluated 997/997 candidates in 854.1s (0.86s/candidate, 32 workers)

Top candidates by ORACLE Dice:
rank      score      bd  oracle       SN   del%   tumSN  params
   1   -9993.02  0.3610  0.6982  1600645   0.1% 42305.6  ψ=3 α=8 βmin=0 βmax=n^0.96 m=[0,255]
   2   -9993.02  0.3610  0.6982  1600645   0.1% 42305.6  ψ=3 α=10 βmin=0 βmax=n^0.96 m=[0,255]
   3   -9993.02  0.3610  0.6982  1600645   0.1% 42305.6  ψ=3 α=15 βmin=0 βmax=n^0.96 m=[0,255]
   4   -9993.02  0.3610  0.6982  1600645   0.1% 42305.6  ψ=3 α=21 βmin=0 βmax=n^0.96 m=[0,255]
   5   -9993.02  0.3610  0.6982  1600645   0.1% 42305.6  ψ=3 α=30 βmin=0 βmax=n^0.96 m=[0,255]
   6   -9994.42  0.3616  0.5583  1335413   0.1% 34121.2  ψ=4 α=32 βmin=0 βmax=n^0.94 m=[8,242]
   7   -9994.61  0.3583  0.5389  1600646   0.0% 42305.8  ψ=3 α=8 βmin=0 βmax=n^1.00 m=[0,255]
   8   -9994.61  0.3583  0.5389  1600646   0.0% 42305.8  ψ=3 α=10 βmin=0 βmax=n^1.00 m=[0,255]
   9   -9994.61  0.3583  0.5389  1600646   0.0% 42305.8  ψ=3 α=15 βmin=0 βmax

## 5. Feature extraction and graph construction

This version uses:

- log volume
- log boundary/surface
- compactness
- elongation
- **principal-axis components** `axis_x, axis_y, axis_z`
- mean intensity
- true intensity std

That yields 9 node features for a single-channel CT volume. The paper lists dominant axis as one descriptor, but the actual axis is a 3D vector, so using its components is the least lossy implementation.


In [7]:
# Luke's feature extraction functions (from rust_crate.ipynb)

def _layout(C):
    return dict(area=0, s=[1, 2, 3],
                cov=[(4, 0, 0), (5, 1, 1), (6, 2, 2), (7, 0, 1), (8, 0, 2), (9, 1, 2)],
                chan0=10, boundary=10 + C + 6, D=3)


def node_invariants(node_feats, C=1, eps=1e-6):
    """Extract scale/orientation-invariant node features from Rust output."""
    f = node_feats.astype(np.float64)
    L = _layout(C)
    D = L["D"]
    N = f.shape[0]
    V = f[:, L["area"]]
    Vsafe = np.maximum(V, 1.0)
    mean_coord = np.stack([f[:, c] for c in L["s"]], axis=1) / Vsafe[:, None]

    cov = np.zeros((N, D, D))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vsafe - mean_coord[:, i] * mean_coord[:, j]
        cov[:, i, j] = cij; cov[:, j, i] = cij

    w = np.linalg.eigvalsh(cov)
    w = np.clip(w, 0.0, None)
    _, vec = np.linalg.eigh(cov)
    principal = vec[..., -1]
    trace = w.sum(axis=1)
    degenerate = trace < eps

    denom = w[:, 2] + eps
    shape = np.stack([(w[:, 2] - w[:, 1]) / denom,
                      (w[:, 1] - w[:, 0]) / denom,
                      w[:, 0] / denom], axis=1)
    shape[degenerate] = 0.0
    line_like = np.where(degenerate, 0.0, shape[:, 0])

    chan = f[:, L["chan0"]:L["chan0"] + C] / Vsafe[:, None] / 255.0
    compactness = f[:, L["boundary"]] / np.power(Vsafe, (D - 1.0) / D)
    elongation = np.where(w[:, 0] > eps, w[:, 2] / (w[:, 0] + eps), 1.0)
    elongation = np.clip(elongation, 1.0, 100.0)

    return dict(V=V, surface=f[:, L["boundary"]], centroid=mean_coord,
                eig=w, shape=shape, line_like=line_like, principal=principal,
                chan=chan, compactness=compactness, elongation=elongation,
                log_size=np.log(Vsafe))


def edge_invariants(node_feats, edge_index, edge_feats, C=1, eps=1e-6):
    """Extract scale-invariant edge features from Rust output."""
    inv = node_invariants(node_feats, C, eps)
    a = edge_index[0].astype(np.int64)
    b = edge_index[1].astype(np.int64)
    ef = edge_feats.astype(np.float64)
    blsafe = np.maximum(ef[:, 0], 1.0)

    size_contrast = np.abs(inv["V"][a] - inv["V"][b]) / (inv["V"][a] + inv["V"][b] + eps)
    bfrac_a = ef[:, 0] / (inv["surface"][a] + eps)
    bfrac_b = ef[:, 0] / (inv["surface"][b] + eps)
    mean_contrast = np.abs(inv["chan"][a] - inv["chan"][b])
    shape_dissim = np.abs(inv["shape"][a] - inv["shape"][b])
    axis_align = (np.abs(np.sum(inv["principal"][a] * inv["principal"][b], axis=1))
                  * np.minimum(inv["line_like"][a], inv["line_like"][b]))
    bcontrast = (ef[:, 1] / blsafe) / 255.0
    cut_frac = ef[:, 3] / blsafe

    cols = [size_contrast[:, None], bfrac_a[:, None], bfrac_b[:, None],
            mean_contrast if mean_contrast.ndim > 1 else mean_contrast[:, None],
            shape_dissim, axis_align[:, None], bcontrast[:, None], cut_frac[:, None]]
    return np.concatenate(cols, axis=1).astype(np.float32)

In [8]:
import torch
from torch_geometric.data import Data


def build_pyg_graph(raw_nf, raw_ei, raw_ef, labels_np, seg, ct_u8, C=1, overlap_threshold=OVERLAP_THRESHOLD):
    """Build a PyG graph from the Rust minor output.

    Labels are assigned by majority/overlap threshold on each supernode. The final
    metric is still lifted voxel Dice, so this threshold should be swept if oracle
    is good but training Dice is low.
    """
    n_sn = raw_nf.shape[0]
    n_edges = raw_ei.shape[1]

    inv = node_invariants(raw_nf, C)
    int_std = compute_intensity_std(labels_np, ct_u8)
    if len(int_std) < n_sn:
        int_std = np.pad(int_std, (0, n_sn - len(int_std)))
    int_std = int_std[:n_sn]

    principal = inv["principal"].astype(np.float32)
    # Eigenvectors have arbitrary sign. Canonicalize sign so the largest absolute
    # component is positive; this reduces random sign flips across cases.
    if len(principal):
        max_comp = np.argmax(np.abs(principal), axis=1)
        signs = np.sign(principal[np.arange(len(principal)), max_comp])
        signs[signs == 0] = 1
        principal = principal * signs[:, None]

    x = np.column_stack([
        np.log1p(inv["V"]),             # volume
        np.log1p(inv["surface"]),       # boundary/surface
        inv["compactness"],             # compactness
        inv["elongation"],              # elongation
        principal[:, 0],                 # dominant axis x
        principal[:, 1],                 # dominant axis y
        principal[:, 2],                 # dominant axis z
        inv["chan"][:, 0],              # mean intensity
        int_std,                         # intensity std
    ]).astype(np.float32)

    # Per-graph z-score normalization.
    for col in range(x.shape[1]):
        mu, sigma = float(x[:, col].mean()), float(x[:, col].std())
        if sigma > 1e-8:
            x[:, col] = (x[:, col] - mu) / sigma
        else:
            x[:, col] = 0.0

    if n_edges > 0:
        edge_attr = edge_invariants(raw_nf, raw_ei, raw_ef, C)
        ei_fwd = torch.tensor(raw_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr_t = torch.tensor(np.concatenate([edge_attr, edge_attr]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr_t = torch.zeros((0, 10), dtype=torch.float32)

    flat = labels_np.ravel()
    valid = flat >= 0
    gt = (seg.ravel() == 2).astype(np.float64)
    max_id = int(flat[valid].max()) if valid.any() else -1
    y = np.zeros(n_sn, dtype=np.int64)
    if max_id >= 0:
        tumor_count = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
        total_count = np.bincount(flat[valid], minlength=max_id + 1)
        overlap = tumor_count / np.maximum(total_count, 1)
        y[:min(n_sn, len(overlap))] = (overlap[:n_sn] >= overlap_threshold).astype(np.int64)

    return Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=edge_index,
        edge_attr=edge_attr_t,
        y=torch.tensor(y, dtype=torch.long),
    )


# Build graphs for all volumes using selected Θopt.
graphs = {}
raw_data = {}
oracles, sns, del_pcts = [], [], []

print(f"Building graphs with Θopt: ψ={MERGE_DIST}, α={CUT_DIST}, βmin={DELETE_SMALL}, βmax=n^{DELETE_LARGE_FRAC:.2f}, m=[{VALUE_MIN},{VALUE_MAX}]")
for vid in ordered:
    ct_raw, seg, ct_u8 = load_and_convert(vid)
    n_vox = ct_raw.size
    params = dict(psi=MERGE_DIST, alpha=CUT_DIST, beta_min=DELETE_SMALL,
                  beta_max_frac=DELETE_LARGE_FRAC, m_min=VALUE_MIN, m_max=VALUE_MAX)
    raw_nf, raw_ei, raw_ef, raw_labels, raw_adj = run_minor(ct_u8, n_vox, params)
    labels_np = np.asarray(raw_labels)

    data = build_pyg_graph(raw_nf, raw_ei, raw_ef, labels_np, seg, ct_u8, overlap_threshold=OVERLAP_THRESHOLD)
    graphs[vid] = data
    raw_data[vid] = {"labels": labels_np, "seg": seg, "n_sn": raw_nf.shape[0]}

    od, t_sn, del_pct = oracle_dice(labels_np, seg)
    oracles.append(od); sns.append(raw_nf.shape[0]); del_pcts.append(del_pct)
    n_tu = int((data.y == 1).sum())
    n_bg = int((data.y == 0).sum())
    print(f"vol-{vid}: {data.num_nodes:,} nodes ({n_tu} tumor, {n_bg:,} bg), "
          f"oracle={od:.4f}, del_tumor={del_pct:.1f}%, edges={data.num_edges:,}, edge_dim={data.edge_attr.shape[1] if data.edge_attr.numel() > 0 else 0}")

mean_oracle = float(np.mean(oracles))
mean_sn = float(np.mean(sns))
mean_del = float(np.mean(del_pcts))
print("\nGraph diagnostics:")
print(f"  Oracle Dice:    {mean_oracle:.4f} ± {np.std(oracles):.4f}")
print(f"  Mean supernodes:{mean_sn:,.0f} ± {np.std(sns):,.0f}")
print(f"  Tumor deleted:  {mean_del:.2f}%")
print("  Paper LiTS target: ~1,075 ± 297 supernodes")

if mean_oracle < 0.80:
    print("\nSTOP: Oracle is low. The coarsener is still the bottleneck; GINE training cannot reach 0.89 until this improves.")
    print("Recommended: increase N_RANDOM, expand low-psi candidates, test HU window, or relax deletion.")
    print("You may still run training for debugging, but treat the result as non-reproducible.")


/home/ud3d4/.conda/envs/llmft/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building graphs with Θopt: ψ=3, α=8, βmin=0, βmax=n^0.96, m=[0,255]


vol-0: 259,432 nodes (400 tumor, 259,032 bg), oracle=0.0579, del_tumor=0.0%, edges=321,786, edge_dim=10


vol-3: 2,391,187 nodes (410 tumor, 2,390,777 bg), oracle=0.8803, del_tumor=0.0%, edges=2,919,376, edge_dim=10


vol-4: 2,285,046 nodes (205820 tumor, 2,079,226 bg), oracle=0.8547, del_tumor=0.3%, edges=3,025,084, edge_dim=10


...

vol-109: 2,047,758 nodes (10822 tumor, 2,036,936 bg), oracle=0.9201, del_tumor=0.0%, edges=2,616,700, edge_dim=10


vol-112: 2,291,234 nodes (235 tumor, 2,290,999 bg), oracle=0.9305, del_tumor=0.0%, edges=3,166,870, edge_dim=10


vol-118: 1,515,539 nodes (63548 tumor, 1,451,991 bg), oracle=0.9694, del_tumor=0.0%, edges=1,634,858, edge_dim=10


vol-126: 1,576,092 nodes (882 tumor, 1,575,210 bg), oracle=0.9481, del_tumor=0.0%, edges=1,618,800, edge_dim=10


vol-130: 3,019,695 nodes (213149 tumor, 2,806,546 bg), oracle=0.9860, del_tumor=0.0%, edges=3,598,484, edge_dim=10

Graph diagnostics:
  Oracle Dice:    0.7701 ± 0.2840
  Mean supernodes:1,718,492 ± 1,168,702
  Tumor deleted:  0.04%
  Paper LiTS target: ~1,075 ± 297 supernodes

STOP: Oracle is low. The coarsener is still the bottleneck; GINE training cannot reach 0.89 until this improves.
Recommended: increase N_RANDOM, expand low-psi candidates, test HU window, or relax deletion.
You may still run training for debugging, but treat the result as non-reproducible.


## 6. GINE Training

Paper spec: 3-layer GINE, hidden 128, Adam lr=1e-3, patience 10.
Luke's additions: sqrt class weights (capped at 30), patience 30.

In [9]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm


class GINE(nn.Module):
    """3-layer GINE — paper spec, with dynamic node/edge dimensions."""
    def __init__(self, node_dim, edge_dim, hidden=128):
        super().__init__()
        self.edge_proj = nn.Linear(edge_dim, hidden)

        def mlp(d_in):
            return nn.Sequential(
                nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                nn.Linear(hidden, hidden),
            )

        self.conv1 = GINEConv(mlp(node_dim), edge_dim=hidden)
        self.bn1 = BatchNorm(hidden)
        self.conv2 = GINEConv(mlp(hidden), edge_dim=hidden)
        self.bn2 = BatchNorm(hidden)
        self.conv3 = GINEConv(mlp(hidden), edge_dim=hidden)
        self.bn3 = BatchNorm(hidden)
        self.head = nn.Linear(hidden, 2)

    def forward(self, data):
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        if ea is not None and ea.numel() > 0:
            ea = self.edge_proj(ea)
        else:
            n = x.size(0)
            ei = torch.stack([torch.arange(n, device=x.device)] * 2)
            ea = torch.zeros(n, self.edge_proj.out_features, device=x.device)
        x = F.relu(self.bn1(self.conv1(x, ei, ea)))
        x = F.relu(self.bn2(self.conv2(x, ei, ea)))
        x = F.relu(self.bn3(self.conv3(x, ei, ea)))
        return self.head(x)


In [10]:

RUN_GINE_TRAINING = mean_oracle >= 0.75
if not RUN_GINE_TRAINING:
    print(f"Skipping GINE training because mean_oracle={mean_oracle:.4f} < 0.75.")
    print("Fix the graph minor first. Set RUN_GINE_TRAINING=True manually only for debugging.")
else:
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    train_graphs = [graphs[v] for v in train_ids]
    val_graphs = [graphs[v] for v in val_ids]

    node_dim = train_graphs[0].x.shape[1]
    sample_ea = train_graphs[0].edge_attr
    edge_dim = sample_ea.shape[1] if sample_ea.numel() > 0 else 10
    print(f"Node dim: {node_dim}; edge dim: {edge_dim}")

    model = GINE(node_dim=node_dim, edge_dim=edge_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    total_pos = sum(int((g.y == 1).sum()) for g in train_graphs)
    total_neg = sum(int((g.y == 0).sum()) for g in train_graphs)
    raw_ratio = total_neg / max(total_pos, 1)
    eff_ratio = min(np.sqrt(raw_ratio), 30.0)
    weight = torch.tensor([1.0, eff_ratio], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    print(f"Class weight: [1.0, {eff_ratio:.1f}] (raw ratio: {raw_ratio:.1f})")
    print(f"Tumor SN: {total_pos:,}; Background SN: {total_neg:,}")


    def lifted_voxel_dice_for_vids(model, vids, device):
        """Compute dataset-level lifted voxel Dice."""
        model.eval()
        tp = fp = fn = 0
        with torch.no_grad():
            for vid in vids:
                data = graphs[vid]
                rd = raw_data[vid]
                try:
                    logits = model(data.to(device))
                except RuntimeError:
                    torch.cuda.empty_cache()
                    continue
                preds = logits.argmax(dim=1).cpu().numpy()
                labels_np = rd["labels"]
                seg = rd["seg"]
                flat = labels_np.ravel()
                valid = flat >= 0
                pred_mask = np.zeros(labels_np.shape, dtype=bool)
                if valid.any():
                    max_id = int(flat[valid].max())
                    lut = np.zeros(max_id + 1, dtype=np.int8)
                    lut[:min(len(preds), max_id + 1)] = preds[:min(len(preds), max_id + 1)]
                    pred_mask = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
                gt_mask = seg == 2
                inter = int((pred_mask & gt_mask).sum())
                tp += inter
                fp += int(pred_mask.sum()) - inter
                fn += int(gt_mask.sum()) - inter
        return 2 * tp / (2 * tp + fp + fn + 1e-8)


    PATIENCE = 30
    best_dice, best_state, wait = -1.0, None, 0
    history = {"train_loss": [], "val_voxel_dice": []}

    for epoch in range(1, 201):
        model.train()
        total_loss = 0.0
        for idx in np.random.permutation(len(train_graphs)):
            g = train_graphs[idx]
            if g.num_nodes > 100000:
                continue
            try:
                g = g.to(device)
                opt.zero_grad()
                loss = criterion(model(g), g.y)
                loss.backward()
                opt.step()
                total_loss += float(loss.item())
            except RuntimeError as e:
                if 'out of memory' in str(e):
                    torch.cuda.empty_cache()
                    continue
                raise
        mean_loss = total_loss / max(len(train_graphs), 1)
        history["train_loss"].append(mean_loss)

        val_dice = lifted_voxel_dice_for_vids(model, val_ids, device)
        history["val_voxel_dice"].append(val_dice)

        if val_dice > best_dice:
            best_dice = val_dice
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stop epoch {epoch}, best lifted val Dice={best_dice:.4f}")
                break

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}  loss={mean_loss:.4f}  lifted_val_dice={val_dice:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\nBest lifted voxel val Dice: {best_dice:.4f}")


Device: cuda:0
Node dim: 9; edge dim: 10


Class weight: [1.0, 9.7] (raw ratio: 93.7)
Tumor SN: 1,462,086; Background SN: 137,059,895


Epoch   1  loss=0.0000  lifted_val_dice=0.0446


Epoch  10  loss=0.0000  lifted_val_dice=0.0446


Epoch  20  loss=0.0000  lifted_val_dice=0.0446


Epoch  30  loss=0.0000  lifted_val_dice=0.0446


Early stop epoch 31, best lifted val Dice=0.0446

Best lifted voxel val Dice: 0.0446


## 7. Voxel-Level Evaluation

Lift supernode predictions to voxel grid via the label volume. Each voxel inherits its supernode's prediction.

In [11]:

if 'model' not in globals() or not RUN_GINE_TRAINING:
    print("No trained GINE model available. Saving graph-only diagnostics summary.")
    results = []
else:
    model.eval()
    model = model.to(device)
    results = []

    for vid in ordered:
        data = graphs[vid]
        rd = raw_data[vid]
        labels_np = rd["labels"]
        seg = rd["seg"]

        with torch.no_grad():
            try:
                preds = model(data.to(device)).argmax(dim=1).cpu().numpy()
            except RuntimeError:
                torch.cuda.empty_cache()
                preds = model.cpu()(data).argmax(dim=1).numpy()
                model = model.to(device)

        flat = labels_np.ravel()
        valid = flat >= 0
        pred_mask = np.zeros(labels_np.shape, dtype=bool)
        if valid.any():
            max_id = int(flat[valid].max())
            lut = np.zeros(max_id + 1, dtype=np.int8)
            lut[:min(len(preds), max_id + 1)] = preds[:min(len(preds), max_id + 1)]
            pred_mask = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)

        gt_mask = seg == 2
        inter = int((gt_mask & pred_mask).sum())
        dice = 2.0 * inter / (gt_mask.sum() + pred_mask.sum() + 1e-8)
        recall = inter / (gt_mask.sum() + 1e-8)
        precision = inter / (pred_mask.sum() + 1e-8) if pred_mask.sum() > 0 else 0.0

        split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
        results.append({"vid": vid, "split": split, "dice": dice, "recall": recall, "precision": precision})
        print(f"vol-{vid} [{split}]: Dice={dice:.4f}  Recall={recall:.4f}  Prec={precision:.4f}")

    print("\n" + "=" * 60)
    for split in ["train", "val", "test"]:
        scores = [r["dice"] for r in results if r["split"] == split]
        if scores:
            print(f"{split:>5s}: Dice = {np.mean(scores):.4f} ± {np.std(scores):.4f}  (n={len(scores)})")

print(f"\nOracle Dice:     {mean_oracle:.4f}")
print(f"Mean supernodes: {mean_sn:.0f}")
print(f"Tumor deleted:   {mean_del:.2f}%")
print("Paper target:    Dice 0.891 ± 0.007, ~1075 SN")

summary = {
    "theta": {"psi": MERGE_DIST, "alpha": CUT_DIST, "beta_min": DELETE_SMALL,
              "beta_max_frac": DELETE_LARGE_FRAC, "m_min": VALUE_MIN, "m_max": VALUE_MAX,
              "overlap_threshold": OVERLAP_THRESHOLD, "hu_min": HU_MIN, "hu_max": HU_MAX},
    "mean_oracle": mean_oracle,
    "mean_supernodes": mean_sn,
    "mean_tumor_deleted_pct": mean_del,
    "gine_training_ran": bool(RUN_GINE_TRAINING),
    "results": results,
}
with open("semir_lits_v4_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved semir_lits_v4_run_summary.json")


vol-0 [train]: Dice=0.0029  Recall=0.8935  Prec=0.0015


vol-3 [train]: Dice=0.0004  Recall=1.0000  Prec=0.0002


vol-4 [train]: Dice=0.1594  Recall=0.8635  Prec=0.0878


vol-5 [train]: Dice=0.0001  Recall=0.8649  Prec=0.0001


vol-6 [train]: Dice=0.0059  Recall=0.9445  Prec=0.0029


...

vol-109 [test]: Dice=0.0089  Recall=0.7895  Prec=0.0045


vol-112 [test]: Dice=0.0002  Recall=0.9909  Prec=0.0001


vol-118 [test]: Dice=0.0850  Recall=0.9651  Prec=0.0445


vol-126 [test]: Dice=0.0010  Recall=0.9938  Prec=0.0005


vol-130 [test]: Dice=0.1278  Recall=0.9565  Prec=0.0685

train: Dice = 0.0158 ± 0.0284  (n=82)
  val: Dice = 0.0318 ± 0.0355  (n=17)
 test: Dice = 0.0287 ± 0.0422  (n=19)

Oracle Dice:     0.7701
Mean supernodes: 1718492
Tumor deleted:   0.04%
Paper target:    Dice 0.891 ± 0.007, ~1075 SN
Saved semir_lits_v4_run_summary.json


## 8. How to interpret this run

The graph-only diagnostics decide whether training is meaningful.

- If `Oracle Dice < 0.75`, do **not** chase GINE training. Expand the search, lower `ψ`, lower/disable `β_min`, disable intensity deletion, or test the HU window.
- If `Tumor deleted > 5%`, deletion is too aggressive. Keep `β_min` small and `m=[0,255]`.
- If `Oracle Dice > 0.85` but `Mean supernodes` is huge, then start trading compression against oracle.
- If `Oracle Dice > 0.85` and `Mean supernodes` is roughly 1k–5k, then train GINE and sweep `OVERLAP_THRESHOLD` across `[0.01, 0.05, 0.10, 0.25, 0.50]`.

This notebook saves `semir_lits_v4_run_summary.json`.
